# Autodiff from scratch

Backpropagation is the chain rule applied to a graph. This notebook builds a tiny reverse-mode autodiff engine in plain Python, checks it against numerical gradients, then uses it to train a small model. No torch or tensorflow needed.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)

## 1. Intuition

Write a calculation as a graph. Each node is one simple operation: add, multiply, square. The forward pass computes values from inputs to output. The backward pass walks the graph in reverse and hands each node the answer to 'how much does the output change if you nudge me?'.

Reverse mode is the right choice for ML because one backward pass gives the gradient with respect to every parameter, and a loss has one output.

## 2. Maths

For a node $v$ feeding into $z$, the chain rule gives $\bar v = \partial z / \partial v$ (the 'adjoint'). If $v$ is used by several nodes, the adjoints add:

$$\bar v = \sum_{c \in \text{children}(v)} \bar c \cdot \frac{\partial c}{\partial v}$$

Local rules: for $c = a + b$, $\partial c/\partial a = 1$. For $c = a b$, $\partial c/\partial a = b$. For $c = a^k$, $\partial c/\partial a = k a^{k-1}$. For $c = \tanh a$, $\partial c/\partial a = 1 - c^2$.

Algorithm: sort the graph topologically, set $\bar z = 1$, visit nodes in reverse order and push each adjoint to its parents.

## 3. Cost

The backward pass visits each node once, so it costs about the same as the forward pass. Gradients for $n$ parameters cost roughly 1 backward pass, against $2n$ forward passes for finite differences. Memory is the price: every intermediate value must be kept for the backward pass.

## 4. From scratch

In [ ]:
class Value:
    def __init__(self, data, parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._backward = lambda: None
        self.op = op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data**k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")

        def _backward():
            self.grad += e * out.grad

        out._backward = _backward
        return out

    def backward(self):
        order, seen = [], set()

        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    visit(p)
                order.append(v)

        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    __radd__ = __add__
    __rmul__ = __mul__

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __truediv__(self, other):
        return self * (other**-1 if isinstance(other, Value) else 1 / other)

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

### Check against a numerical gradient

Use the function from the old TensorFlow notebook, $f(w_1, w_2) = 3 w_1^2 + 2 w_1 w_2$ at $(5, 3)$. By hand: $\partial f/\partial w_1 = 6 w_1 + 2 w_2 = 36$ and $\partial f/\partial w_2 = 2 w_1 = 10$.

In [ ]:
def f(w1, w2):
    return 3 * w1**2 + 2 * w1 * w2


a, b = Value(5.0), Value(3.0)
z = f(a, b)
z.backward()
print("autodiff:", a.grad, b.grad)

eps = 1e-6
print(
    "numeric: ",
    (f(5 + eps, 3) - f(5 - eps, 3)) / (2 * eps),
    (f(5, 3 + eps) - f(5, 3 - eps)) / (2 * eps),
)

A node used twice must add its gradients, not overwrite them. In $x \cdot x$ the gradient is $2x$, so with $x = 4$ it should be 8.

In [ ]:
x = Value(4.0)
y = x * x
y.backward()
print("d(x*x)/dx:", x.grad)

### A reusable gradient checker

Run the check on a messier function that uses `tanh`, `exp`, division and a shared node.

In [ ]:
def grad_check(fn, inputs, eps=1e-6):
    vals = [Value(v) for v in inputs]
    fn(*vals).backward()
    auto = np.array([v.grad for v in vals])
    num = []
    for i in range(len(inputs)):
        up, dn = list(inputs), list(inputs)
        up[i] += eps
        dn[i] -= eps
        num.append(
            (fn(*[Value(v) for v in up]).data - fn(*[Value(v) for v in dn]).data) / (2 * eps)
        )
    return auto, np.array(num)


def messy(a, b, c):
    s = a * b + c
    return (s.tanh() + (a * 0.5).exp()) / (b * b + 1) + s * s


auto, num = grad_check(messy, [0.7, -1.3, 0.4])
print("autodiff:", auto)
print("numeric: ", num)
print("max abs diff:", np.abs(auto - num).max())
assert np.allclose(auto, num, atol=1e-6)

### Use it: fit a line with gradient descent

Fit $y = 2 + 3x$ with the same loss as before, mean squared error, but with gradients from the engine.

In [ ]:
xs = rng.uniform(-1, 1, size=50)
ys = 2.0 + 3.0 * xs + 0.1 * rng.normal(size=50)

w, b = Value(0.0), Value(0.0)
history = []
for step in range(100):
    w.grad = b.grad = 0.0
    loss = Value(0.0)
    for x_i, y_i in zip(xs, ys):
        loss = loss + (w * x_i + b - y_i) ** 2
    loss = loss * (1 / len(xs))
    loss.backward()
    w.data -= 0.3 * w.grad
    b.data -= 0.3 * b.grad
    history.append(loss.data)

print("w, b:", round(w.data, 3), round(b.data, 3), " final loss:", round(history[-1], 4))
plt.figure(figsize=(4, 3))
plt.plot(history)
plt.xlabel("step")
plt.ylabel("MSE")
plt.show()

## 5. Library version

PyTorch and TensorFlow do the same thing on tensors, with a C++ backend. They are optional extras in this project (`uv sync --extra torch`) so the cell below is guarded. The core path does not need them.

In [ ]:
try:
    import torch

    w1 = torch.tensor(5.0, requires_grad=True)
    w2 = torch.tensor(3.0, requires_grad=True)
    f(w1, w2).backward()
    print("torch:", w1.grad.item(), w2.grad.item())
except ImportError:
    print("torch not installed, skipping. Expected 36.0 and 10.0.")

Notes on the library API. PyTorch builds the graph as you run, and `backward()` frees it afterwards. TensorFlow's `GradientTape` records operations inside a `with` block and erases the tape after one `gradient()` call unless `persistent=True`. Both are the same algorithm you just wrote.

## 6. What breaks

1. Forgetting to zero gradients between steps. They accumulate, so the update is wrong.
2. Reusing a graph. Here, rebuilding it each step is what keeps things correct.
3. Non-differentiable points. $|x|$ at 0 and ReLU at 0 have no true gradient. Libraries pick a convention.
4. Deep chains. Multiplying many local derivatives below 1 shrinks the gradient towards zero.

In [ ]:
# 3 above: stale gradients
x = Value(2.0)
for _ in range(3):
    (x * x).backward()  # no reset
print("after 3 backward calls without reset:", x.grad, "(correct value is 4.0)")

# 4 above: vanishing gradient through 30 tanh layers
h = Value(0.5)
x0 = h
for _ in range(30):
    h = (h * 0.8).tanh()
h.backward()
print("gradient through 30 tanh layers:", f"{x0.grad:.2e}")

Thirty layers cut the gradient to under $10^{-3}$ of its starting scale. Section 05 returns to this when it covers why ReLU and residual connections help.